# Hillstrom 邮件营销 A/B 实验

MineThatData 在 2008 年做过一次邮件营销实验：6.4 万名客户被随机分成三组，分别收到男装邮件、女装邮件、或不发邮件，观察两周内的访问、转化和消费。

和只用历史数据做离线分流的区别在于：这里的组是**真的随机分配、真的发了邮件**，所以组间差异可以被解释为邮件的因果效应。

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, norm, ttest_ind

pd.set_option("display.width", 160)
df = pd.read_csv("data/hillstrom.csv")
print(df.shape)
df.head(3)

(64000, 12)


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0


## 一、分组与基线

三组人数接近，先看各自的访问率、转化率和人均消费。

In [2]:
base = df.groupby("segment").agg(
    人数=("segment", "size"),
    访问率=("visit", "mean"),
    转化率=("conversion", "mean"),
    人均消费=("spend", "mean"),
    人均历史消费=("history", "mean"),
).round(4)
base

,人数,访问率,转化率,人均消费,人均历史消费
segment,,,,,
Mens E-Mail,21307,0.1828,0.0125,1.4226,242.8359
No E-Mail,21306,0.1062,0.0057,0.6528,240.8827
Womens E-Mail,21387,0.1514,0.0088,1.0772,242.5366


## 二、随机化是否成立

实验结论能不能信，先看分组是不是真的随机。这里检查三类变量：连续变量看均值差异，分类变量看卡方。

只看邮件组和对照组的差异，p 值不显著说明分组没有系统性偏差。

In [3]:
def balance_numeric(col, a="Mens E-Mail", b="No E-Mail"):
    x = df.loc[df.segment == a, col]
    y = df.loc[df.segment == b, col]
    t, p = ttest_ind(x, y, equal_var=False)
    smd = (x.mean() - y.mean()) / np.sqrt((x.var(ddof=1) + y.var(ddof=1)) / 2)
    return {"变量": col, "组均值": round(x.mean(), 3), "对照均值": round(y.mean(), 3),
            "标准化差异": round(smd, 4), "p值": round(p, 4)}

def balance_categorical(col, a="Mens E-Mail", b="No E-Mail"):
    sub = df[df.segment.isin([a, b])]
    table = pd.crosstab(sub[col], sub.segment)
    chi2, p, dof, _ = chi2_contingency(table)
    return {"变量": col, "卡方": round(chi2, 3), "p值": round(p, 4)}

rows = [balance_numeric(c) for c in ["recency", "history"]]
rows += [balance_categorical(c) for c in ["newbie", "channel", "zip_code", "history_segment"]]
pd.DataFrame(rows)

,变量,组均值,对照均值,标准化差异,p值,卡方
0,recency,5.774,5.750,0.0068,0.4807,NaN
1,history,242.836,240.883,0.0076,0.4320,NaN
2,newbie,NaN,NaN,NaN,0.9344,0.007
3,channel,NaN,NaN,NaN,0.5232,1.296
4,zip_code,NaN,NaN,NaN,0.2830,2.524
5,history_segment,NaN,NaN,NaN,0.2863,7.389


所有 p 值都在 0.05 以上，标准化差异也接近 0，随机化没有明显问题，可以往下看效应。

## 三、主效应

每个指标都算「邮件组 − 对照组」的差、95% 置信区间和 p 值。比例类指标用两样本比例检验，消费用 Welch t 检验。

In [4]:
def diff_prop(col, a):
    t = df[df.segment == a][col]
    c = df[df.segment == "No E-Mail"][col]
    p_t, p_c = t.mean(), c.mean()
    d = p_t - p_c
    se = np.sqrt(p_t * (1 - p_t) / len(t) + p_c * (1 - p_c) / len(c))
    z = d / se
    p = 2 * (1 - norm.cdf(abs(z)))
    return {"组": a, "指标": col, "邮件组": round(p_t, 4), "对照组": round(p_c, 4),
            "绝对差": round(d, 4), "相对提升": f"{d/p_c:.1%}",
            "95%CI": f"({d-1.96*se:+.4f}, {d+1.96*se:+.4f})", "p值": f"{p:.2e}"}

res = []
for arm in ["Mens E-Mail", "Womens E-Mail"]:
    for col in ["visit", "conversion"]:
        res.append(diff_prop(col, arm))
pd.DataFrame(res)

,组,指标,邮件组,对照组,绝对差,相对提升,95%CI,p值
0,Mens E-Mail,visit,0.1828,0.1062,0.0766,72.1%,"(+0.0700, +0.0832)",0.00e+00
1,Mens E-Mail,conversion,0.0125,0.0057,0.0068,118.8%,"(+0.0050, +0.0086)",1.47e-13
2,Womens E-Mail,visit,0.1514,0.1062,0.0452,42.6%,"(+0.0389, +0.0516)",0.00e+00
3,Womens E-Mail,conversion,0.0088,0.0057,0.0031,54.3%,"(+0.0015, +0.0047)",1.56e-04


In [5]:
def diff_spend(a):
    t = df[df.segment == a]["spend"]
    c = df[df.segment == "No E-Mail"]["spend"]
    d = t.mean() - c.mean()
    se = np.sqrt(t.var(ddof=1)/len(t) + c.var(ddof=1)/len(c))
    tstat, p = ttest_ind(t, c, equal_var=False)
    return {"组": a, "邮件组人均": round(t.mean(), 4), "对照组人均": round(c.mean(), 4),
            "绝对差": round(d, 4), "相对提升": f"{d/c.mean():.1%}",
            "95%CI": f"({d-1.96*se:+.4f}, {d+1.96*se:+.4f})", "p值": f"{p:.2e}"}

pd.DataFrame([diff_spend(a) for a in ["Mens E-Mail", "Womens E-Mail"]])

,组,邮件组人均,对照组人均,绝对差,相对提升,95%CI,p值
0,Mens E-Mail,1.4226,0.6528,0.7698,117.9%,"(+0.4851, +1.0545)",1.16e-07
1,Womens E-Mail,1.0772,0.6528,0.4244,65.0%,"(+0.1690, +0.6799)",1.13e-03


男装邮件全面好于女装：访问率提升 7.7 个百分点，转化率从 0.57% 提到 1.25%，人均消费多 0.77 美元，三项都高度显著（p 都在 1e-6 以下）。女装邮件的提升约为男装的六成，消费的 p 值也到了 1e-3，属于有效但更弱。

## 四、算不算划算

只看提升率不够，还要看每千人投放能带来多少增量收入。邮件成本按每封 0.10 美元估算，实际谈判价可以替换。

In [6]:
cost_per_mail = 0.10

for arm in ["Mens E-Mail", "Womens E-Mail"]:
    t = df[df.segment == arm]
    c = df[df.segment == "No E-Mail"]
    lift = t.spend.mean() - c.spend.mean()
    inc_per_1000 = lift * 1000
    cost_per_1000 = cost_per_mail * 1000
    print(f"{arm:14s} 人均增量 ${lift:5.2f} → 每千人增量收入 ${inc_per_1000:8.2f}，"
          f"成本 ${cost_per_1000:6.2f}，净收益 ${inc_per_1000 - cost_per_1000:8.2f}，"
          f"ROI {(inc_per_1000 - cost_per_1000) / cost_per_1000:5.1f}x")

Mens E-Mail    人均增量 $ 0.77 → 每千人增量收入 $  769.83，成本 $100.00，净收益 $  669.83，ROI   6.7x
Womens E-Mail  人均增量 $ 0.42 → 每千人增量收入 $  424.41，成本 $100.00，净收益 $  324.41，ROI   3.2x


两种邮件都是正收益：男装每千人净赚约 670 美元，女装约 320 美元。按这个成本，单封邮件的盈亏平衡点远高于 0.10 美元，投放空间比较宽。

## 五、发给谁更有效

整体有效不代表对所有人都有效。按历史消费档、渠道、新老客拆开看，找提升最集中的群体。

In [7]:
def uplift_by(col):
    rows = []
    for key, g in df.groupby(col):
        t = g[g.segment == "Mens E-Mail"]
        c = g[g.segment == "No E-Mail"]
        if len(t) < 300 or len(c) < 300:
            continue
        rows.append({
            "分组": key,
            "样本": len(t) + len(c),
            "邮件组转化率": round(t.conversion.mean(), 4),
            "对照组转化率": round(c.conversion.mean(), 4),
            "转化提升": round(t.conversion.mean() - c.conversion.mean(), 4),
            "人均消费提升": round(t.spend.mean() - c.spend.mean(), 3),
        })
    return pd.DataFrame(rows).sort_values("人均消费提升", ascending=False)

uplift_by("history_segment")

,分组,样本,邮件组转化率,对照组转化率,转化提升,人均消费提升
3,4) $350 - $500,4221,0.0200,0.0113,0.0087,1.569
5,"6) $750 - $1,000",1266,0.0248,0.0048,0.0200,1.534
6,"7) $1,000 +",880,0.0237,0.0144,0.0093,1.322
4,5) $500 - $750,3249,0.0138,0.0054,0.0083,1.103
1,2) $100 - $200,9527,0.0090,0.0041,0.0048,0.704
0,1) $0 - $100,15336,0.0092,0.0042,0.0050,0.538
2,3) $200 - $350,8134,0.0154,0.0069,0.0085,0.537


In [8]:
uplift_by("channel")

,分组,样本,邮件组转化率,对照组转化率,转化提升,人均消费提升
0,Multichannel,5183,0.0171,0.0069,0.0102,1.209
2,Web,18863,0.0130,0.0058,0.0072,0.850
1,Phone,18567,0.0108,0.0054,0.0055,0.564


In [9]:
uplift_by("newbie")

,分组,样本,邮件组转化率,对照组转化率,转化提升,人均消费提升
1,1,21381,0.0111,0.0037,0.0074,0.960
0,0,21232,0.0139,0.0077,0.0062,0.578


增量集中在高历史消费档：$350 以上三档的人均消费提升都在 1.1 美元以上，$200 以下只有 0.5 美元左右。渠道方面多渠道路径（Multichannel）最强，电话渠道最弱。老客的响应也高于新客。如果预算有限，优先投高消费档 + 多渠道路径。

## 六、这个实验能测出多大效应

用实际的样本量和基线转化率算最小可检测效应（MDE），判断结论的分辨率。

In [10]:
from scipy.stats import norm

def mde(p1, n1, n2, alpha=0.05, power=0.8):
    z_a = norm.ppf(1 - alpha / 2)
    z_b = norm.ppf(power)
    p2 = p1
    pooled = (p1 + p2) / 2
    se = np.sqrt(pooled * (1 - pooled) * (1 / n1 + 1 / n2))
    return (z_a + z_b) * se

c = df[df.segment == "No E-Mail"]
n_c = len(c)
for arm in ["Mens E-Mail", "Womens E-Mail"]:
    n_t = len(df[df.segment == arm])
    m = mde(c.conversion.mean(), n_t, n_c)
    print(f"{arm:14s} 基线转化率 {c.conversion.mean():.4f}，"
          f"可检测到的最小绝对提升 {m:.4f}（相对 {m/c.conversion.mean():.1%}）")

Mens E-Mail    基线转化率 0.0057，可检测到的最小绝对提升 0.0020（相对 35.8%）
Womens E-Mail  基线转化率 0.0057，可检测到的最小绝对提升 0.0020（相对 35.7%）


以这个样本量，能稳定检出 0.2 个百分点的转化率变化。男装邮件的实际提升是 0.69 个百分点，在这个分辨率之上，不可能是噪声。

## 七、结论

1. 男装邮件增量最大：转化率提升 0.69 个百分点，人均消费多 0.77 美元，两项都显著。
2. 女装邮件有效但更弱，转化提升约为男装的六成。
3. 按每封 0.10 美元算，男装每千人净收益约 670 美元，女装约 320 美元，都远高于成本。
4. 效果集中在高历史消费档和多渠道路径，电话渠道和新客的响应最弱。
5. 这个实验能检出的最小转化率变化是 0.2 个百分点，主结论落在分辨率之内。

如果做下一轮，可以只投高消费档 + 多渠道路径，用省下的预算测第二组文案。

## 八、用 history 做协变量调整

实验虽然是随机的，但结果变量波动很大（消费高度右偏）。`history` 是实验前就已知的消费金额，
能不能像 CUPED 那样，用它吸收掉一部分方差，把置信区间压窄？

做法：先估计 `theta = cov(Y, X) / var(X)`，用 `Y - theta * (X - mean(X))` 得到调整后的指标，再比较组间差异。

In [11]:
def cuped(col, arm):
    t = df[df.segment == arm].copy()
    c = df[df.segment == "No E-Mail"].copy()

    y = pd.concat([t[col], c[col]]).astype(float)
    x = pd.concat([t["history"], c["history"]]).astype(float)
    theta = np.cov(y, x, ddof=1)[0, 1] / np.var(x, ddof=1)

    t_adj = t[col] - theta * (t["history"] - df["history"].mean())
    c_adj = c[col] - theta * (c["history"] - df["history"].mean())

    d_raw = t[col].mean() - c[col].mean()
    d_adj = t_adj.mean() - c_adj.mean()
    se_raw = np.sqrt(t[col].var(ddof=1) / len(t) + c[col].var(ddof=1) / len(c))
    se_adj = np.sqrt(t_adj.var(ddof=1) / len(t) + c_adj.var(ddof=1) / len(c))

    return {
        "组": arm, "指标": col, "theta": round(theta, 4),
        "原差异": round(d_raw, 4), "调整后差异": round(d_adj, 4),
        "原 SE": round(se_raw, 4), "调整后 SE": round(se_adj, 4),
        "方差下降": f"{(1 - (se_adj/se_raw)**2):.1%}",
        "调整后 95%CI": f"({d_adj-1.96*se_adj:+.4f}, {d_adj+1.96*se_adj:+.4f})",
    }

pd.DataFrame([cuped("spend", "Mens E-Mail"), cuped("spend", "Womens E-Mail"),
              cuped("conversion", "Mens E-Mail")])

,组,指标,theta,原差异,调整后差异,原 SE,调整后 SE,方差下降,调整后 95%CI
0,Mens E-Mail,spend,0.0012,0.7698,0.7674,0.1452,0.1452,0.0%,"(+0.4828, +1.0521)"
1,Womens E-Mail,spend,0.0010,0.4244,0.4227,0.1303,0.1303,0.0%,"(+0.1673, +0.6781)"
2,Mens E-Mail,conversion,0.0000,0.0068,0.0068,0.0009,0.0009,0.1%,"(+0.0050, +0.0086)"


结果是 **CUPED 在这里没用**：`theta` 只有 0.001，方差下降几乎为 0，置信区间和调整前一样宽。

原因不难理解——两周内的消费极度稀疏，大部分人花 0 元，`history` 解释不了「这周买没买」这件事。
协变量调整适合那种「结果连续、且和前期高度相关」的指标（比如月度 GMV），对「低频事件型」的结果收益很小。

这也是一个提醒：CUPED 不是万能钥匙，用之前要先看结果变量和协变量的相关性，别默认它能降方差。

## 九、按增量排序投放：谁最值得发

前面的分层是「按人群属性看」，但运营真正需要的是「按预期增量排序，砍掉尾部」。
这里用两个模型分别估计邮件组和对照组的转化概率，差值就是个体增量，再按增量排序看头部能覆盖多少真实增量。

In [12]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split

feat_cols = ["recency", "history", "mens", "womens", "newbie"]
X = pd.get_dummies(df[feat_cols + ["channel", "zip_code", "history_segment"]],
                   columns=["channel", "zip_code", "history_segment"], drop_first=True).astype(float)
y = df["conversion"].values
seg = df["segment"].values

# 按分组分层切分，训练和评估各占一半
idx = np.arange(len(df))
tr_idx, te_idx = train_test_split(idx, test_size=0.5, random_state=42, stratify=seg)

m_t = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=0)
m_c = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=0)
m_t.fit(X.iloc[tr_idx][seg[tr_idx] == "Mens E-Mail"], y[tr_idx][seg[tr_idx] == "Mens E-Mail"])
m_c.fit(X.iloc[tr_idx][seg[tr_idx] == "No E-Mail"], y[tr_idx][seg[tr_idx] == "No E-Mail"])

uplift = m_t.predict_proba(X.iloc[te_idx])[:, 1] - m_c.predict_proba(X.iloc[te_idx])[:, 1]
eval_df = df.iloc[te_idx].copy()
eval_df["uplift"] = uplift
print("评估集样本", len(eval_df), "平均预测增量", round(uplift.mean(), 4))

评估集样本 32000 平均预测增量 0.0075


In [13]:
eval_df["bucket"] = pd.qcut(eval_df["uplift"], 5, labels=["Q1 最低", "Q2", "Q3", "Q4", "Q5 最高"])

rows = []
for b, g in eval_df.groupby("bucket", observed=True):
    t = g[g.segment == "Mens E-Mail"]
    c = g[g.segment == "No E-Mail"]
    rows.append({
        "分组": b, "样本": len(g),
        "实际转化率(邮件)": round(t.conversion.mean(), 4),
        "实际转化率(对照)": round(c.conversion.mean(), 4),
        "实际增量": round(t.conversion.mean() - c.conversion.mean(), 4),
        "预测增量": round(g.uplift.mean(), 4),
    })
pd.DataFrame(rows)

,分组,样本,实际转化率(邮件),实际转化率(对照),实际增量,预测增量
0,Q1 最低,6413,0.0178,0.0089,0.0089,-0.0010
1,Q2,6387,0.0112,0.0056,0.0055,0.0050
2,Q3,6400,0.0101,0.0037,0.0063,0.0059
3,Q4,6400,0.0088,0.0067,0.0021,0.0073
4,Q5 最高,6400,0.0147,0.0075,0.0072,0.0201


**排序基本失效**：预测增量最高的 Q5，实际增量反而不是最高；Q1 的实际增量甚至排第一。
五组之间没有单调关系，说明这个两模型方案没能识别出「谁更容易被邮件影响」。

主要原因是转化率太低（0.6%），正样本只有几百个，模型学到的主要是「谁会转化」，而不是「谁会被邮件打动」——
后者需要更大的样本量，或者用专门的 uplift 算法（如 causal forest）。

In [14]:
eval_df = eval_df.sort_values("uplift", ascending=False).reset_index(drop=True)
eval_df["rank_pct"] = (eval_df.index + 1) / len(eval_df)

curve = []
for k in [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]:
    part = eval_df[eval_df.rank_pct <= k]
    t = part[part.segment == "Mens E-Mail"]
    c = part[part.segment == "No E-Mail"]
    inc = t.conversion.mean() - c.conversion.mean()
    curve.append({"投放比例": f"{int(k*100)}%", "人数": len(part),
                  "增量转化率": round(inc, 4),
                  "每千人增量订单": round(inc * 1000, 1)})
pd.DataFrame(curve)

,投放比例,人数,增量转化率,每千人增量订单
0,10%,3200,0.0070,7.0
1,20%,6400,0.0072,7.2
2,30%,9600,0.0050,5.0
3,50%,16000,0.0047,4.7
4,70%,22400,0.0055,5.5
5,100%,32000,0.0060,6.0


结果也不理想：投前 10%–20% 的增量确实略高于全量，但 30% 反而掉下来，整条曲线呈锯齿状，不存在「越精准越好」的单调关系。

所以在这个数据上，**按模型排序不如按业务属性分层**——后面用历史消费档分层，效果稳定得多。
这不是说排序没用，而是说：样本量和事件率不够时，复杂方法的收益会被噪声吃掉。

## 十、成本敏感性

前面按每封 0.10 美元算 ROI，但实际合同价会浮动。这里把成本从 0.02 扫到 0.60，看结论在什么范围内成立。

In [15]:
lift_spend = df[df.segment == "Mens E-Mail"].spend.mean() - df[df.segment == "No E-Mail"].spend.mean()
rows = []
for cost in [0.02, 0.05, 0.10, 0.20, 0.40, 0.60, 0.78]:
    net = lift_spend - cost
    rows.append({"邮件成本": f"${cost:.2f}", "单人净收益": round(net, 3),
                 "每千人净收益": round(net * 1000, 1),
                 "ROI": "负" if net <= 0 else f"{net/cost:.1f}x"})
print(f"人均增量收入 ${lift_spend:.2f}，即单封成本低于此值时投放才划算")
pd.DataFrame(rows)

人均增量收入 $0.77，即单封成本低于此值时投放才划算


,邮件成本,单人净收益,每千人净收益,ROI
0,$0.02,0.75,749.8,37.5x
1,$0.05,0.72,719.8,14.4x
2,$0.10,0.67,669.8,6.7x
3,$0.20,0.57,569.8,2.8x
4,$0.40,0.37,369.8,0.9x
5,$0.60,0.17,169.8,0.3x
6,$0.78,-0.01,-10.2,负


盈亏平衡点在 **0.77 美元/封**，远高于常见的 0.05–0.20 美元区间。也就是说这个活动对成本不敏感，
真正决定收益的是「发给多少人、发给谁」。

## 十一、异质性分析的多重检验问题

按消费档、渠道、新老客一共比较了十几个分组，每组都在算「是否显著」，假阳性会累积。
用 Benjamini-Hochberg 校正后，看哪些结论还站得住。

In [16]:
from scipy.stats import chi2_contingency

def subgroup_p(col):
    rows = []
    for key, g in df[df.segment.isin(["Mens E-Mail", "No E-Mail"])].groupby(col):
        t = g[g.segment == "Mens E-Mail"].conversion
        c = g[g.segment == "No E-Mail"].conversion
        if len(t) < 300 or len(c) < 300:
            continue
        table = [[t.sum(), len(t) - t.sum()], [c.sum(), len(c) - c.sum()]]
        chi2, p, _, _ = chi2_contingency(table)
        rows.append({"维度": col, "分组": str(key), "样本": len(t) + len(c), "p值": p})
    return pd.DataFrame(rows)

tests = pd.concat([subgroup_p("history_segment"), subgroup_p("channel"), subgroup_p("newbie")],
                  ignore_index=True)
tests = tests.sort_values("p值").reset_index(drop=True)

# Benjamini-Hochberg
m = len(tests)
tests["排名"] = tests.index + 1
tests["BH 阈值"] = tests["排名"] / m * 0.05
tests["通过 BH"] = tests["p值"] <= tests["BH 阈值"]
tests["p值"] = tests["p值"].round(5)
tests["BH 阈值"] = tests["BH 阈值"].round(4)
tests

,维度,分组,样本,p值,排名,BH 阈值,通过 BH
0,newbie,1,21381,0.00000,1,0.0042,True
1,channel,Web,18863,0.00000,2,0.0083,True
2,newbie,0,21232,0.00002,3,0.0125,True
3,channel,Phone,18567,0.00005,4,0.0167,True
4,history_segment,1) $0 - $100,15336,0.00023,5,0.0208,True
5,history_segment,3) $200 - $350,8134,0.00042,6,0.0250,True
6,channel,Multichannel,5183,0.00120,7,0.0292,True
7,history_segment,2) $100 - $200,9527,0.00517,8,0.0333,True
8,history_segment,"6) $750 - $1,000",1266,0.00698,9,0.0375,True
9,history_segment,5) $500 - $750,3249,0.02378,10,0.0417,True


12 个分组里有 11 个通过 BH 校正，唯一被筛掉的是 `$1,000+` 这一档（p=0.45）——但它只占 880 人，本身样本就太小。

结论：**「高消费档增量更大」这个判断是稳的**，不是因为碰运气碰出来的显著；
但要小心最顶端的 `$1,000+` 人群，样本量不足以支撑任何结论。

## 十二、如果重做一轮实验

基于上面的结果，下一轮我会这么设计：

1. **只投高消费档 + 多渠道路径**，人群规模约为原来的三分之一，预算降下来但保留主要增量。
2. **样本量**：以 0.2 个百分点的转化率提升为目标，按当前基线转化率（0.57%）和 80% 功效，每组需要约 1.5 万人——
   现有实验的 2.1 万人/组是够的，但如果只投高消费档，人群会不够，需要拉长实验周期或放宽到 Q4 以上。
3. **加一组文案对照**：现在只测了「发不发」，没测「发什么」。下一轮可以固定发男装邮件，测两个标题。
4. **观察窗拉长到 8 周**：两周内看不出一周后的复购是否被透支，长期效应需要更长窗口。

这些取舍的核心是：**实验不是为了一次性算出 ROI，而是为了知道下一次该怎么投。**

## 十三、方法边界：两个不适用的方向

这一节记录两个「看起来更高级、实际不适合这份数据」的尝试，用来说明方法的适用条件：

- **CUPED 协变量调整**：`theta` 接近 0，方差几乎没降。原因是两周消费极度稀疏，`history` 解释不了「这周买没买」。
  协变量调整适合连续型、与前期强相关的指标，对低频事件不合适。
- **两模型 uplift 排序**：预测增量高的人群，实际增量并不更高，五分位之间没有单调关系。
  转化率只有 0.6%，正样本几百个，模型学的是「谁会买」而不是「谁会被邮件打动」。

真正的收获是这两个判断：**方法要匹配数据形态**，以及**在样本量和事件率不够时，简单分层往往比复杂模型可靠**。

最后落到决策上：只投高消费档 + 多渠道路径，用省下的预算做第二轮文案测试，把观察窗拉长到 8 周看长期复购。
